In [46]:
import pandas as pd
import requests
import os

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import clean_adzuna
import merge_jobs

from config import (
    APP_ID,
    APP_KEY,
    COUNTRY,
    RESULTS_PER_PAGE,
    UPDATE_PAGES,
    LATEST_ADZUNA,
    ADZUNA_CSV,      
    MASTER_FILE,
)

from logger import logger

In [47]:
from config import MASTER_FILE

print(MASTER_FILE)

C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs.csv


In [48]:
BASE_URL = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search"

In [49]:
#Download one page of jobs from Adzuna
def fetch_page(session, page):

    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": RESULTS_PER_PAGE,
        "sort_by": "date",
    }

    url = f"{BASE_URL}/{page}"

    try:

        response = session.get(
            url,
            params=params,
            timeout=REQUEST_TIMEOUT,
        )

        response.raise_for_status()

        data = response.json()

        jobs = data.get("results", [])

        logger.info(f"Downloaded page {page}: {len(jobs)} jobs")

        return jobs

    except requests.RequestException as error:

        logger.error(f"Page {page} failed: {error}")

        return []

In [50]:
# Load all existing Adzuna job IDs from the master dataset

def get_existing_ids():
    
    if not os.path.exists(MASTER_FILE):
        logger.info("No master_jobs.csv found. First run.")
        return set()

    df = pd.read_csv(MASTER_FILE)

    if "source" not in df.columns or "source_id" not in df.columns:
        logger.warning("master_jobs.csv is missing 'source' or 'id' columns.")
        return set()

    existing_ids = set(
    df[df["source"] == "Adzuna"]["source_id"].astype(str)
)

    logger.info(f"Loaded {len(existing_ids)} existing IDs.")

    return existing_ids

In [51]:
#Create a requests session with automatic retries.

def create_session():
    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    session = requests.Session()

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session.mount("https://", adapter)

    return session

In [52]:
#test session
session = create_session()
print(session)

In [53]:
# Test the Adzuna API connection.
def test_connection(session):
    

    url = f"{BASE_URL}/1"

    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 5,
        "sort_by": "date",
    }

    response = session.get(
        url,
        params=params,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    data = response.json()

    return data

In [54]:
#testing
session = create_session()

data = test_connection(session)

print(data.keys())

dict_keys(['__CLASS__', 'results', 'count', 'mean'])


In [55]:
# test the Job data is being retrieved
print(f"Number of jobs returned: {len(data['results'])}")

first_job = data['results'][0]

print(first_job['title'])
print(first_job['company']['display_name'])
print(first_job['location']['display_name'])

Number of jobs returned: 5
Health And Safety Manager
Go2personnel Solutions
Easton, Bristol


In [56]:
#Collect only NEW jobs from Adzuna

def collect_jobs(session):
   
    existing_ids = get_existing_ids()

    new_jobs = []

    logger.info("Checking for new jobs...")

    for page in range(1, UPDATE_PAGES + 1):

        jobs = fetch_page(session, page)

        if not jobs:
            logger.info(f"Page {page}: No jobs returned.")
            break

        page_new = 0

        for job in jobs:

            job_id = str(job.get("id"))

            if job_id not in existing_ids:

                new_jobs.append(job)

                existing_ids.add(job_id)

                page_new += 1

        logger.info(f"Page {page}: {page_new} new jobs")

        if page_new == 0:
            logger.info("No more new jobs found.")
            break

    logger.info(f"Collected {len(new_jobs)} new jobs.")

    return new_jobs

In [57]:
#Save downloaded jobs to CSV.
def save_jobs(jobs):

    if not jobs:
        logger.info("No new jobs to save.")
        return

    df = pd.DataFrame(jobs)

    df.to_csv(LATEST_ADZUNA, index=False)

    logger.info(f"Saved {len(df)} new jobs to {LATEST_ADZUNA}")

In [58]:
#Append newly downloaded jobs to the historical Adzuna dataset.
def update_raw_dataset():
    
    incoming = pd.read_csv(LATEST_ADZUNA)

    if ADZUNA_CSV.exists():

        historical = pd.read_csv(ADZUNA_CSV)

        df = pd.concat([historical, incoming], ignore_index=True)

        df = df.drop_duplicates(subset="id")

    else:
        df = incoming

    df.to_csv(ADZUNA_CSV, index=False)

    logger.info(f"Historical dataset updated: {len(df)} jobs")

In [59]:
def main():

    logger.info("===== Adzuna Update Started =====")
    session = create_session()

    jobs = collect_jobs(session)

    # If there are no new jobs, stop here
    if not jobs:
        logger.info("No new jobs found.")
        return

    # Save raw jobs
    save_jobs(jobs)

    save_jobs(jobs)
    
# Clean the downloaded data
    logger.info("Updating historical Adzuna dataset...")
    update_raw_dataset()

    logger.info("Running Adzuna cleaning...")
    clean_adzuna.clean_adzuna()

    logger.info("Updating master dataset...")
    merge_jobs.merge_jobs()

    logger.info("Adzuna pipeline completed successfully.")

    logger.info("===== Adzuna Update Completed =====")


if __name__ == "__main__":
    main()

2026-07-25 17:16:10 | INFO | ===== Adzuna Update Started =====
2026-07-25 17:16:11 | INFO | Loaded 6525 existing IDs.
2026-07-25 17:16:11 | INFO | Checking for new jobs...
2026-07-25 17:16:13 | INFO | Downloaded page 1: 50 jobs
2026-07-25 17:16:13 | INFO | Page 1: 0 new jobs
2026-07-25 17:16:13 | INFO | No more new jobs found.
2026-07-25 17:16:13 | INFO | Collected 0 new jobs.
2026-07-25 17:16:13 | INFO | No new jobs found.
